Celda 1: Importación de librerías y configuración de la GPU

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Configuración para evitar problemas de memoria en la GPU
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU configurada y lista para la acción.")
    except RuntimeError as e:
        print(e)

2026-06-06 13:31:23.189547: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-06 13:31:23.573359: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-06 13:31:23.573864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-06 13:31:23.638694: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-06 13:31:23.770949: I tensorflow/core/platform/cpu_feature_guar

GPU configurada y lista para la acción.


2026-06-06 13:31:29.024922: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.275194: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.275345: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.275358: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


Celda 2: Carga de metadatos y Blindaje contra el Data Leakage

In [2]:
# Cambia las rutas según tu entorno de trabajo (Kaggle, Colab o Local)
METADATA_PATH = "HAM10000_metadata.csv"
IMAGES_DIR = "all_images/"  # Carpeta donde juntaste las imágenes de la parte 1 y 2

df = pd.read_csv(METADATA_PATH)

# Creamos la columna con la ruta exacta de cada imagen
df["path"] = df["image_id"].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))

# Mapeamos las etiquetas de texto a números enteros
label_map = {
    "nv": 0,  # Nevus melanocíticos (Mayoritaria)
    "mel": 1,  # Melanoma
    "bcl": 2,  # Carcinoma basocelular
    "akiec": 3,  # Queratosis actínica
    "vasc": 4,  # Lesiones vasculares (Minoritaria)
    "df": 5,  # Dermatofibroma (Minoritaria)
    "akiec": 6,  # Queratosis benignas
}
# Asegúrate de verificar las clases exactas en tu CSV, ajustamos las 7 categorías típicas:
label_mapping = {label: idx for idx, label in enumerate(df["dx"].unique())}
df["label"] = df["dx"].map(label_mapping)
num_classes = len(label_mapping)

print("Distribución original de clases:")
print(df["dx"].value_counts())

# --- EL TRUCO CLAVE: Separación por Lesion ID ---
# Extraemos los lesion_id únicos junto con su clase correspondiente
unique_lesions = df.drop_duplicates(subset="lesion_id").copy()

# Primero, separamos un 15% para el Test definitivo
train_val_lesions, test_lesions = train_test_split(
    unique_lesions, test_size=0.15, stratify=unique_lesions["label"], random_state=42
)

# Ahora, del resto, separamos un 15% para Validación
train_lesions, val_lesions = train_test_split(
    train_val_lesions,
    test_size=0.15,
    stratify=train_val_lesions["label"],
    random_state=42,
)

# Mapeamos los IDs de vuelta al dataframe original para recuperar TODAS las imágenes asociadas
train_df = df[df["lesion_id"].isin(train_lesions["lesion_id"])].sample(
    frac=1, random_state=42
)
val_df = df[df["lesion_id"].isin(val_lesions["lesion_id"])].sample(
    frac=1, random_state=42
)
test_df = df[df["lesion_id"].isin(test_lesions["lesion_id"])].sample(
    frac=1, random_state=42
)

print(
    f"\nSeparación limpia realizada:\nTrain: {len(train_df)} imágenes\nVal: {len(val_df)} imágenes\nTest: {len(test_df)} imágenes"
)

Distribución original de clases:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Separación limpia realizada:
Train: 7225 imágenes
Val: 1280 imágenes
Test: 1510 imágenes


Celda 3: Construcción del Pipeline de Datos Ultrarrápido (tf.data)

In [3]:
IMG_SIZE = 224  # Tamaño óptimo para EfficientNetB2
BATCH_SIZE = 32


def load_and_preprocess_image(path, label):
    # Leer el archivo
    image = tf.io.read_file(path)
    # Decodificar JPG
    image = tf.image.decode_jpeg(image, channels=3)
    # Redimensionar de forma limpia
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # EfficientNet de Keras incluye su propia capa de reescalado interna,
    # pero nos aseguramos de que los tipos de datos sean flotantes.
    image = tf.cast(image, tf.float32)
    return image, label


# Aumentación de datos agresiva para combatir el desbalance en entrenamiento
data_augmentation = models.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.3),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.15),
    ]
)


def configure_dataset(df, augment=False):
    paths = df["path"].values
    labels = df["label"].values

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    # Mezclamos y cargamos las imágenes en paralelo (Multithreading nativo)
    dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.map(
        load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    if augment:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    return dataset.prefetch(buffer_size=tf.data.AUTOTUNE)


train_ds = configure_dataset(train_df, augment=True)
val_ds = configure_dataset(val_df, augment=False)
test_ds = configure_dataset(test_df, augment=False)

2026-06-06 13:31:29.363729: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.363823: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.363859: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 13:31:29.363866: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-06-06 13:31:29.886372: I external/lo

Celda 4: La Función de Pérdida Salvavidas (Categorical Focal Loss Custom)

In [4]:
# Diseñamos una Focal Loss personalizada para clasificación multiclase.
# Castiga severamente al modelo si se confía demasiado en la clase mayoritaria.
def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        # Evitar ceros y desbordamientos numéricos
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        # Convertir y_true a float32 si no lo está
        y_true = tf.cast(y_true, tf.float32)

        # Calcular Cross Entropy tradicional
        cross_entropy = -y_true * tf.math.log(y_pred)

        # Calcular el factor de modulación (Focal Term)
        loss = alpha * tf.math.pow(1.0 - y_pred, gamma) * cross_entropy

        # Sumar a través de las clases y promediar en el batch
        return tf.math.reduce_sum(loss, axis=-1)

    return focal_loss_fixed

Celda 5: Arquitectura del Modelo (Transfer Learning con EfficientNetB2)

In [5]:
# Cargamos la base preentrenada en ImageNet, congelando las capas iniciales
base_model = tf.keras.applications.EfficientNetB2(
    weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # Congelamos temporalmente para la fase de calentamiento

# Añadimos la cabeza de clasificación personalizada
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)  # Dropout alto para frenar en seco el overfitting
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)

# Compilamos usando un optimizador Adam clásico y nuestra Focal Loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"],
)

model.summary()

31790344/31790344 [==============================] - 2s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 efficientnetb2 (Functional  (None, 7, 7, 1408)        7768569   
 )                                                               
                                                                 
 global_average_pooling2d (  (None, 1408)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 batch_normalization (Batch  (None, 1408)              5632      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 1408)              0      

Celda 6: Entrenamiento por Fases (Calentamiento y Fine-Tuning Avanzado)

In [6]:
# Callbacks estratégicos para no quemar tu hardware y guardar lo mejor
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6
    ),
]

print("--- FASE 1: Entrenando la cabeza de clasificación ---")
history_phase1 = model.fit(
    train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks
)

# --- FASE 2: Fine-Tuning Descongelando Capas ---
print("\n--- FASE 2: Descongelando el modelo para ajuste fino ---")
base_model.trainable = True

# Regla de oro: Al hacer Fine-Tuning, usa un Learning Rate extremadamente bajo
# para no destrozar los pesos preentrenados que ya funcionan bien.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=categorical_focal_loss(gamma=2.5, alpha=0.25),  # Aumentamos gamma ligeramente
    metrics=["accuracy"],
)

history_phase2 = model.fit(
    train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks
)

--- FASE 1: Entrenando la cabeza de clasificación ---
Epoch 1/10


2026-06-06 13:31:41.429480: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-06-06 13:31:41.495941: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:225] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 12.0
2026-06-06 13:31:41.496047: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:228] Used ptxas at /home/jesus/repos/TFM/.venv/lib/python3.11/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc/bin/ptxas
2026-06-06 13:31:41.496134: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-06-06 13:31:42.412489: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-06-06 13:31:42.430928: W tensorflow/compiler

226/226 [==============================] - 33s 105ms/step - loss: 5.6559 - accuracy: 0.1433 - val_loss: 4.1102 - val_accuracy: 0.1406 - lr: 0.0010
Epoch 2/10
226/226 [==============================] - 23s 102ms/step - loss: 4.5707 - accuracy: 0.1434 - val_loss: 4.0402 - val_accuracy: 0.0945 - lr: 0.0010
Epoch 3/10
226/226 [==============================] - 24s 104ms/step - loss: 4.1163 - accuracy: 0.1528 - val_loss: 3.9567 - val_accuracy: 0.1594 - lr: 0.0010
Epoch 4/10
226/226 [==============================] - 24s 103ms/step - loss: 3.9236 - accuracy: 0.1457 - val_loss: 3.9131 - val_accuracy: 0.1203 - lr: 0.0010
Epoch 5/10
226/226 [==============================] - 24s 103ms/step - loss: 3.8600 - accuracy: 0.1528 - val_loss: 3.8957 - val_accuracy: 0.1375 - lr: 0.0010
Epoch 6/10
226/226 [==============================] - 24s 104ms/step - loss: 3.8429 - accuracy: 0.1477 - val_loss: 3.8911 - val_accuracy: 0.0820 - lr: 0.0010
Epoch 7/10
226/226 [==============================] - 24s 105ms

2026-06-06 13:36:05.541247: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.76GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-06-06 13:36:05.625283: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.76GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-06-06 13:36:05.718419: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-06-06 13:36:05.720303: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compi

226/226 [==============================] - 115s 328ms/step - loss: 3.5528 - accuracy: 0.1345 - val_loss: 3.6011 - val_accuracy: 0.1531 - lr: 1.0000e-05
Epoch 2/15
226/226 [==============================] - 73s 322ms/step - loss: 3.5528 - accuracy: 0.1607 - val_loss: 3.6011 - val_accuracy: 0.1336 - lr: 1.0000e-05
Epoch 3/15
226/226 [==============================] - 73s 322ms/step - loss: 3.5528 - accuracy: 0.1682 - val_loss: 3.6011 - val_accuracy: 0.1508 - lr: 1.0000e-05
Epoch 4/15
226/226 [==============================] - 71s 311ms/step - loss: 3.5528 - accuracy: 0.1493 - val_loss: 3.6011 - val_accuracy: 0.1375 - lr: 1.0000e-05
Epoch 5/15
226/226 [==============================] - 71s 311ms/step - loss: 3.5528 - accuracy: 0.1557 - val_loss: 3.6011 - val_accuracy: 0.1469 - lr: 2.0000e-06
Epoch 6/15
226/226 [==============================] - 71s 314ms/step - loss: 3.5528 - accuracy: 0.1446 - val_loss: 3.6011 - val_accuracy: 0.1945 - lr: 2.0000e-06
Epoch 7/15
226/226 [==================

Celda 7: Evaluación sin piedad de las Clases Minoritarias

In [7]:
print("\n--- EVALUACIÓN FINAL EN EL CONJUNTO DE TEST BLINDADO ---")

# Extraemos las etiquetas reales del set de prueba
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

# Generamos el reporte de clasificación detallado por clase
target_names = [
    key for key, value in sorted(label_mapping.items(), key=lambda item: item[1])
]
print(classification_report(y_true, y_pred, target_names=target_names))

# Matriz de confusión para ver dónde le duele al modelo
cm = confusion_matrix(y_true, y_pred)
print("Matriz de Confusión:")
print(cm)


--- EVALUACIÓN FINAL EN EL CONJUNTO DE TEST BLINDADO ---
              precision    recall  f1-score   support

         bkl       0.08      0.07      0.08       173
          nv       0.62      0.15      0.25      1002
          df       0.00      0.00      0.00        17
         mel       0.15      0.33      0.20       177
        vasc       0.01      0.05      0.01        20
         bcc       0.06      0.03      0.04        71
       akiec       0.03      0.26      0.05        50

    accuracy                           0.16      1510
   macro avg       0.13      0.13      0.09      1510
weighted avg       0.44      0.16      0.20      1510

Matriz de Confusión:
[[ 12  36  11  53  14   4  43]
 [100 155  52 230  95  26 344]
 [  0   3   0  11   0   0   3]
 [ 14  36   8  59  31   2  27]
 [  6   0   2   4   1   0   7]
 [  8  11   3  32   8   2   7]
 [  3  11   2  14   6   1  13]]
